# Dataset and cohort characterisation

Exploratory summary of the accepted RNA-seq cohorts. PCA is descriptive only and is not part of the predictive modelling pipeline.

In [1]:
from pathlib import Path
import hashlib
import json
import csv
import io
import os
from datetime import datetime, timezone
from zipfile import ZipFile

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from sklearn.decomposition import PCA

try:
    from google.colab import drive
    drive.mount("/content/drive")
except ImportError:
    pass

PROJECT = Path("/content/drive/MyDrive/CMPS355_Research")
if PROJECT.exists():
    DATA = PROJECT / "data/validated/ibd_dataset_sample_aligned.zip"
    OUTPUT_ROOT = PROJECT / "runs/cohort_characterization"
else:
    matches = list(Path("/kaggle/input").rglob("ibd_dataset_sample_aligned.zip")) if Path("/kaggle/input").exists() else []
    if len(matches) != 1:
        raise FileNotFoundError("Expected one accepted dataset archive in Drive or Kaggle inputs")
    DATA = matches[0]
    OUTPUT_ROOT = Path("/kaggle/working/cmps355/cohort_characterization")

EXPECTED_SHA256 = "8a83f20035fcac5ae5d8bfe496e4d7491e47400b61ca1080b828c8ad54e91a74"
actual_sha = hashlib.sha256(DATA.read_bytes()).hexdigest()
if actual_sha != EXPECTED_SHA256:
    raise ValueError("Dataset checksum mismatch")

run_id = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")
OUT = OUTPUT_ROOT / run_id
OUT.mkdir(parents=True, exist_ok=False)
print("Dataset integrity: PASS")
print("Results:", OUT)


Mounted at /content/drive
Dataset integrity: PASS
Results: /content/drive/MyDrive/CMPS355_Research/runs/cohort_characterization/20261107T090645Z


In [2]:
cohorts = ("d1", "d2", "d3", "d4")
records = []
matrices = []
shared_genes = None

with ZipFile(DATA) as zf:
    if zf.testzip() is not None:
        raise ValueError("Corrupted dataset archive")
    for cohort in cohorts:
        with zf.open(f"count_{cohort}_aligned.csv") as handle:
            header = next(csv.reader(io.TextIOWrapper(handle, encoding="utf-8-sig")))
        gene_set = set(header[1:])
        if not header or header[0] != "ID" or len(gene_set) != len(header) - 1:
            raise ValueError(f"Invalid gene header for {cohort}")
        shared_genes = gene_set if shared_genes is None else shared_genes.intersection(gene_set)
    if len(shared_genes) != 9983:
        raise ValueError("Unexpected shared-gene count")
    ordered_genes = sorted(shared_genes)
    for cohort, expected in zip(cohorts, (260,179,76,174)):
        desc = pd.read_csv(zf.open(f"desc_{cohort}.csv"), dtype={"ID": str, "class": str, "Study": str})
        counts = pd.read_csv(zf.open(f"count_{cohort}_aligned.csv"))
        if len(desc) != expected or len(counts) != expected or desc["ID"].tolist() != counts["ID"].tolist():
            raise ValueError(f"Sample alignment mismatch in {cohort}")
        if set(desc["class"]) != {"CD", "Not IBD"}:
            raise ValueError(f"Unexpected class labels in {cohort}")
        all_counts = counts.iloc[:, 1:].to_numpy(dtype=np.float64)
        shared_counts = counts.loc[:, ordered_genes].to_numpy(dtype=np.float64)
        if not np.isfinite(all_counts).all() or np.any(all_counts < 0) or np.any(all_counts.sum(axis=1) <= 0):
            raise ValueError(f"Invalid read counts in {cohort}")
        cohort_rows = desc[["ID", "Study", "class"]].copy()
        cohort_rows.insert(0, "cohort", cohort)
        cohort_rows["raw_gene_count"] = all_counts.shape[1]
        cohort_rows["total_count_all_genes"] = all_counts.sum(axis=1).astype(np.int64)
        cohort_rows["total_count_shared_genes"] = shared_counts.sum(axis=1).astype(np.int64)
        cohort_rows["detected_genes_all"] = (all_counts > 0).sum(axis=1)
        cohort_rows["detected_genes_shared"] = (shared_counts > 0).sum(axis=1)
        records.append(cohort_rows)
        matrices.append(shared_counts)

samples = pd.concat(records, ignore_index=True)
if len(samples) != 689 or not samples["ID"].is_unique or samples["Study"].nunique() != 6:
    raise ValueError("Incorrect overall sample metadata")
X = np.vstack(matrices)
if X.shape != (689, 9983):
    raise ValueError("Incorrect shared expression matrix shape")
samples.to_csv(OUT / "table_sample_qc.csv", index=False)

summary = samples.groupby("cohort", sort=True).agg(
    n=("ID", "count"),
    n_studies=("Study", "nunique"),
    n_original_genes=("raw_gene_count", "max"),
    median_total_reads=("total_count_all_genes", "median"),
    median_shared_reads=("total_count_shared_genes", "median"),
    median_detected_genes=("detected_genes_all", "median"),
).reset_index()
counts = samples.groupby(["cohort", "class"]).size().unstack(fill_value=0)
summary["CD"] = summary["cohort"].map(counts["CD"])
summary["Not IBD"] = summary["cohort"].map(counts["Not IBD"])
summary["CD_fraction"] = summary["CD"] / summary["n"]
summary.to_csv(OUT / "table_cohort_characteristics.csv", index=False)

study_table = samples.groupby(["cohort", "Study", "class"]).size().rename("n").reset_index()
study_table.to_csv(OUT / "table_bioproject_composition.csv", index=False)
print(summary.to_string(index=False))


cohort   n  n_studies  n_original_genes  median_total_reads  median_shared_reads  median_detected_genes  CD  Not IBD  CD_fraction
    d1 260          1             10061           5584134.0            5563876.5                 8692.0 218       42     0.838462
    d2 179          2             10061           2259601.0            2256414.0                 7848.0 134       45     0.748603
    d3  76          1             10061           1786881.0            1784091.5                 7753.5  34       42     0.447368
    d4 174          2             41257          22761789.5           12006252.0                21377.0 109       65     0.626437


In [3]:
logcpm = np.log2(1.0 + X * (1_000_000.0 / X.sum(axis=1))[:, None])
variable = logcpm.var(axis=0) > 1e-12
pca = PCA(n_components=2, svd_solver="randomized", random_state=355)
coords = pca.fit_transform(logcpm[:, variable])
points = samples[["ID", "cohort", "Study", "class"]].copy()
points["PC1"] = coords[:, 0]
points["PC2"] = coords[:, 1]
points.to_csv(OUT / "pca_coordinates.csv", index=False)
variance = pd.DataFrame({"component": ["PC1", "PC2"], "explained_variance_ratio": pca.explained_variance_ratio_})
variance.to_csv(OUT / "pca_explained_variance.csv", index=False)

def save_figure(fig, stem):
    fig.savefig(OUT / (stem + ".png"), dpi=300, bbox_inches="tight")
    fig.savefig(OUT / (stem + ".pdf"), bbox_inches="tight")
    plt.close(fig)

fig, ax = plt.subplots(figsize=(7, 4.5))
positions = np.arange(len(summary))
ax.bar(positions, summary["CD"], label="Crohn's disease")
ax.bar(positions, summary["Not IBD"], bottom=summary["CD"], label="Not IBD")
ax.set_xticks(positions, [s.upper() for s in summary["cohort"]])
ax.set_ylabel("Number of samples")
ax.legend(frameon=False)
ax.set_title("Case–control composition by analytical cohort")
save_figure(fig, "figure1_cohort_class_balance")

fig, ax = plt.subplots(figsize=(7, 4.5))
ax.boxplot([samples.loc[samples.cohort == d, "total_count_all_genes"] for d in cohorts], tick_labels=[c.upper() for c in cohorts], showfliers=False)
ax.set_yscale("log")
ax.set_ylabel("Raw counts per sample (all original genes)")
ax.set_title("Sequencing count distributions by cohort")
save_figure(fig, "figure2_raw_library_sizes")

fig, ax = plt.subplots(figsize=(7, 5))
for cohort in cohorts:
    subset = points[points.cohort == cohort]
    ax.scatter(subset.PC1, subset.PC2, s=17, alpha=0.75, label=cohort.upper())
ax.set_xlabel(f"PC1 ({pca.explained_variance_ratio_[0]:.1%} variance)")
ax.set_ylabel(f"PC2 ({pca.explained_variance_ratio_[1]:.1%} variance)")
ax.set_title("Descriptive PCA of shared-gene log-CPM")
ax.legend(frameon=False)
save_figure(fig, "figure3_pca_by_cohort")

fig, ax = plt.subplots(figsize=(7, 5))
for study in sorted(points.Study.unique()):
    subset = points[points.Study == study]
    ax.scatter(subset.PC1, subset.PC2, s=17, alpha=0.75, label=study)
ax.set_xlabel(f"PC1 ({pca.explained_variance_ratio_[0]:.1%} variance)")
ax.set_ylabel(f"PC2 ({pca.explained_variance_ratio_[1]:.1%} variance)")
ax.set_title("Descriptive PCA by source BioProject")
ax.legend(frameon=False, fontsize=8)
save_figure(fig, "figure4_pca_by_bioproject")

manifest = {
    "stage": "cohort_characterization",
    "status": "PASS",
    "dataset_sha256": actual_sha,
    "samples": len(samples),
    "shared_genes": X.shape[1],
    "cohorts": list(cohorts),
    "pca": {"type": "descriptive_only_not_for_model_fitting", "input": "log2(1 + CPM) on shared-gene counts", "svd": "randomized", "random_seed": 355},
    "limitations": [
        "PCA uses all samples for exploratory visualization only and must never be reused as supervised-model input",
        "Observed cohort separation does not by itself establish a causal batch effect",
        "Clinical covariates beyond Study and diagnosis were not evaluated"
    ],
    "created_utc": datetime.now(timezone.utc).isoformat()
}
manifest["artifacts_sha256"] = {p.name: hashlib.sha256(p.read_bytes()).hexdigest() for p in sorted(OUT.iterdir()) if p.is_file()}
(OUT / "report.json").write_text(json.dumps(manifest, indent=2, sort_keys=True), encoding="utf-8")
print("STAGE 4B COHORT CHARACTERIZATION: PASS")
print("Files saved:", len(manifest["artifacts_sha256"]) + 1)
print("Output:", OUT)


STAGE 4B COHORT CHARACTERIZATION: PASS
Files saved: 14
Output: /content/drive/MyDrive/CMPS355_Research/runs/cohort_characterization/20261107T090645Z
